# 01 — EDA & Feature Engineering
## Wind Turbine Gearbox Anomaly Detection

Bu notebook, 5 yıllık SCADA veri setinin kapsamlı keşifsel veri analizini (EDA) ve özellik mühendisliğini içermektedir.

**Dataset:** [Wind Turbine Gearbox Anomaly Detection 5-Year SCADA](https://www.kaggle.com/datasets/aiwithcagri/wind-turbine-gearbox-anomaly-detection-5year-scada/data)

**İçerik:**
1. Dataset yükleme ve genel inceleme
2. Zaman serisi görselleştirme
3. Korelasyon analizi
4. Anomali dönemlerinin görselleştirilmesi
5. Rolling istatistikler
6. Lag features
7. Fourier transform / frekans analizi
8. Feature importance (mutual info)
9. Class imbalance analizi

## 1. Dataset İndirme (Kaggle API)

Aşağıdaki komut ile dataset Kaggle'dan indirilebilir. `kaggle.json` API anahtarının `~/.kaggle/` dizininde olduğundan emin olun.

In [ ]:
# Kaggle API ile dataset indirme
# !pip install kaggle -q
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d aiwithcagri/wind-turbine-gearbox-anomaly-detection-5year-scada
# !unzip -q wind-turbine-gearbox-anomaly-detection-5year-scada.zip -d data/

# Kaggle notebook ortamında:
import os
import glob

# Kaggle ortamında data dosyaları /kaggle/input/ altındadır
DATA_PATH = '/kaggle/input/wind-turbine-gearbox-anomaly-detection-5year-scada/'

# Yerel ortam için:
if not os.path.exists(DATA_PATH):
    DATA_PATH = './data/'

print(f'Data path: {DATA_PATH}')
files = glob.glob(os.path.join(DATA_PATH, '**'), recursive=True)
for f in files[:20]:
    print(f)

## 2. Kütüphaneleri İçe Aktarma

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Görsel ayarlar
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

print('Libraries loaded successfully!')

## 3. Dataset Yükleme ve Genel İnceleme

Dataset'i yükleyip temel istatistikleri inceleyeceğiz: shape, dtypes, missing values ve describe.

In [ ]:
# CSV dosyalarını bul ve yükle
csv_files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
print(f'Found {len(csv_files)} CSV files:')
for f in csv_files:
    print(f'  {os.path.basename(f)}')

# İlk CSV'yi yükle (veya birleştir)
dfs = []
for f in sorted(csv_files):
    df_temp = pd.read_csv(f)
    dfs.append(df_temp)
    print(f'{os.path.basename(f)}: {df_temp.shape}')

df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
print(f'\nTotal dataset shape: {df.shape}')

In [ ]:
# Temel bilgiler
print('=== DATASET INFO ===')
print(f'Shape: {df.shape}')
print(f'\nColumn types:')
print(df.dtypes)
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

In [ ]:
# Zaman damgasını parse et
time_col = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower() or 'timestamp' in c.lower()]
if time_col:
    df[time_col[0]] = pd.to_datetime(df[time_col[0]])
    df = df.sort_values(time_col[0]).reset_index(drop=True)
    df = df.set_index(time_col[0])
    print(f'Time column: {time_col[0]}')
    print(f'Date range: {df.index.min()} → {df.index.max()}')
    print(f'Duration: {(df.index.max() - df.index.min()).days} days')
else:
    print('No time column detected automatically. Please check column names:')
    print(df.columns.tolist())

In [ ]:
# İlk 5 satır
df.head()

In [ ]:
# İstatistiksel özet
print('=== DESCRIPTIVE STATISTICS ===')
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Eksik değer analizi
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing_df) > 0:
    print('Missing values found:')
    print(missing_df)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['Missing %'].plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Missing Value Percentage per Feature')
    ax.set_ylabel('Missing %')
    plt.tight_layout()
    plt.savefig('results/missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No missing values found!')

In [ ]:
# Anomali sütununu belirle
anomaly_col = [c for c in df.columns if 'anomal' in c.lower() or 'label' in c.lower() or 'fault' in c.lower() or 'alarm' in c.lower()]
print(f'Potential anomaly columns: {anomaly_col}')

if anomaly_col:
    ANOMALY_COL = anomaly_col[0]
else:
    # Varsayılan olarak son sütunu al - kullanıcı düzenlemelidir
    ANOMALY_COL = df.columns[-1]
    print(f'Using last column as anomaly target: {ANOMALY_COL}')

print(f'\nAnomaly column: {ANOMALY_COL}')
print(f'Value counts:\n{df[ANOMALY_COL].value_counts()}')

# Sayısal feature sütunları
FEATURE_COLS = df.select_dtypes(include=[np.number]).columns.tolist()
if ANOMALY_COL in FEATURE_COLS:
    FEATURE_COLS.remove(ANOMALY_COL)
print(f'\nFeature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')

## 4. Zaman Serisi Görselleştirme

Her sensörün zaman içindeki davranışını görselleştiriyoruz. Anomali dönemleri kırmızı bölge olarak işaretlenecektir.

In [ ]:
def plot_sensor_with_anomalies(df, sensor_col, anomaly_col, ax, title=None):
    """Sensör verisini anomali dönemleriyle birlikte çizen yardımcı fonksiyon."""
    ax.plot(df.index, df[sensor_col], linewidth=0.5, alpha=0.8, label=sensor_col)
    
    # Anomali dönemlerini şeffaf kırmızı ile işaretle
    anomaly_mask = df[anomaly_col] == 1
    if anomaly_mask.any():
        ax.fill_between(df.index, df[sensor_col].min(), df[sensor_col].max(),
                       where=anomaly_mask, alpha=0.3, color='red', label='Anomaly')
    
    ax.set_title(title or sensor_col)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

# Her sensörü ayrı subplot'ta göster
n_cols = min(2, len(FEATURE_COLS))
n_rows = (len(FEATURE_COLS) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten() if n_rows * n_cols > 1 else [axes]

for i, col in enumerate(FEATURE_COLS):
    plot_sensor_with_anomalies(df, col, ANOMALY_COL, axes[i])

# Boş subplot'ları gizle
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Sensor Time Series with Anomaly Periods', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('results/sensor_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Korelasyon Analizi

Sensörler arasındaki korelasyonu inceliyoruz. Yüksek korelasyon, redundant features veya birlikte hareket eden sensörleri gösterebilir.

In [ ]:
corr_matrix = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, mask=mask, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('results/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Yüksek korelasyonlu çiftler
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.85:
            high_corr.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

if high_corr:
    print('Highly correlated feature pairs (|r| > 0.85):')
    print(pd.DataFrame(high_corr).sort_values('Correlation', key=abs, ascending=False))

## 6. Anomali Timeline Görselleştirmesi

Anomali dönemlerinin 5 yıllık zaman çizelgesi üzerinde işaretlenmesi.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Anomali timeline
axes[0].fill_between(df.index, 0, df[ANOMALY_COL], color='red', alpha=0.7, label='Anomaly')
axes[0].set_title('Anomaly Timeline (5-Year Period)', fontsize=14)
axes[0].set_ylabel('Anomaly Flag')
axes[0].set_yticks([0, 1])
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Aylık anomali yoğunluğu
if hasattr(df.index, 'month'):
    monthly_anomaly = df[ANOMALY_COL].resample('ME').mean() * 100
    monthly_anomaly.plot(kind='bar', ax=axes[1], color='coral', alpha=0.8)
    axes[1].set_title('Monthly Anomaly Rate (%)', fontsize=14)
    axes[1].set_ylabel('Anomaly Rate (%)')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/anomaly_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# Anomali istatistikleri
total = len(df)
anomaly_count = df[ANOMALY_COL].sum()
print(f'Total records: {total:,}')
print(f'Anomaly records: {int(anomaly_count):,} ({anomaly_count/total*100:.2f}%)')
print(f'Normal records: {total - int(anomaly_count):,} ({(total-anomaly_count)/total*100:.2f}%)')

## 7. Rolling İstatistikler (Moving Window Features)

Rolling mean ve rolling std, zaman serilerinde trend ve volatiliteyi yakalamak için kullanılır.
- **window=24**: 24 saatlik (1 günlük) pencere
- **window=48**: 2 günlük pencere
- **window=168**: 1 haftalık pencere (7 gün × 24 saat)

In [ ]:
WINDOWS = [24, 48, 168]
df_features = df.copy()

for col in FEATURE_COLS:
    for w in WINDOWS:
        df_features[f'{col}_roll_mean_{w}'] = df[col].rolling(window=w, min_periods=1).mean()
        df_features[f'{col}_roll_std_{w}'] = df[col].rolling(window=w, min_periods=1).std().fillna(0)

print(f'Rolling features added. New shape: {df_features.shape}')
print(f'New feature count: {df_features.shape[1] - df.shape[1]}')

# İlk sensör için görselleştir
if FEATURE_COLS:
    col = FEATURE_COLS[0]
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    
    sample = df_features.iloc[:2000]  # İlk 2000 gözlem
    axes[0].plot(sample.index, sample[col], label='Original', alpha=0.6, linewidth=0.8)
    for w in WINDOWS:
        axes[0].plot(sample.index, sample[f'{col}_roll_mean_{w}'], label=f'Rolling Mean {w}h', linewidth=1.5)
    axes[0].set_title(f'{col} — Rolling Mean Comparison')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    for w in WINDOWS:
        axes[1].plot(sample.index, sample[f'{col}_roll_std_{w}'], label=f'Rolling Std {w}h', linewidth=1.5)
    axes[1].set_title(f'{col} — Rolling Std Comparison')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/rolling_features.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Lag Features

Lag features, geçmiş zaman adımlarındaki değerleri mevcut adıma taşır. Bu, zaman serisi modellerinin geçmiş bağımlılıkları öğrenmesine yardımcı olur.
- **lag=1**: 1 adım önceki değer
- **lag=6**: 6 saat önceki değer
- **lag=12**: 12 saat önceki değer
- **lag=24**: 24 saat önceki değer

In [ ]:
LAGS = [1, 6, 12, 24]

for col in FEATURE_COLS:
    for lag in LAGS:
        df_features[f'{col}_lag_{lag}'] = df[col].shift(lag)

# NaN değerleri doldur (ilk birkaç satır)
df_features = df_features.bfill().fillna(0)

print(f'Lag features added. New shape: {df_features.shape}')
print(f'Total new features from lag: {len(FEATURE_COLS) * len(LAGS)}')

# Lag korelasyonu görselleştir
if FEATURE_COLS:
    col = FEATURE_COLS[0]
    lag_corrs = []
    for lag in range(1, 49):
        corr = df[col].corr(df[col].shift(lag))
        lag_corrs.append(corr)
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(1, 49), lag_corrs, color='steelblue', alpha=0.7)
    ax.axhline(y=0.05, color='red', linestyle='--', label='Significance threshold')
    ax.axhline(y=-0.05, color='red', linestyle='--')
    ax.set_title(f'Autocorrelation (ACF) — {col}')
    ax.set_xlabel('Lag (hours)')
    ax.set_ylabel('Correlation')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/lag_autocorrelation.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Fourier Transform — Frekans Analizi

FFT (Fast Fourier Transform) ile zaman serisindeki periyodik bileşenleri tespit ediyoruz. Vites kutusu gibi mekanik sistemlerde belirli frekanslardaki güç artışları arıza belirtisi olabilir.

In [ ]:
from scipy import signal

def plot_fft_analysis(series, title, ax_time, ax_freq, color='steelblue'):
    """Zaman serisi ve FFT spektrumunu çizer."""
    # Zaman serisi
    ax_time.plot(series.values[:2000], linewidth=0.5, color=color)
    ax_time.set_title(f'{title} — Time Domain')
    ax_time.set_xlabel('Sample')
    ax_time.grid(True, alpha=0.3)
    
    # FFT
    clean_series = series.dropna().values
    N = len(clean_series)
    fft_vals = np.abs(np.fft.rfft(clean_series - clean_series.mean()))
    freqs = np.fft.rfftfreq(N)
    
    ax_freq.semilogy(freqs[1:N//2], fft_vals[1:N//2], color=color, linewidth=0.8)
    ax_freq.set_title(f'{title} — Frequency Domain (FFT)')
    ax_freq.set_xlabel('Frequency (cycles/sample)')
    ax_freq.set_ylabel('Amplitude (log)')
    ax_freq.grid(True, alpha=0.3)

# İlk 3 sensör için FFT analizi
n_sensors = min(3, len(FEATURE_COLS))
fig, axes = plt.subplots(n_sensors, 2, figsize=(16, n_sensors * 4))
if n_sensors == 1:
    axes = axes.reshape(1, -1)

colors = ['steelblue', 'coral', 'forestgreen']
for i, col in enumerate(FEATURE_COLS[:n_sensors]):
    plot_fft_analysis(df[col], col, axes[i, 0], axes[i, 1], color=colors[i])

plt.suptitle('FFT Frequency Analysis per Sensor', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('results/fft_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fourier özellikleri ekle (dominant frekans bileşenleri)
def add_fourier_features(df_feat, col, n_harmonics=3, window=168):
    """Sliding window FFT ile fourier features üretir."""
    t = np.arange(len(df_feat))
    for k in range(1, n_harmonics + 1):
        df_feat[f'{col}_sin_{k}'] = np.sin(2 * np.pi * k * t / window)
        df_feat[f'{col}_cos_{k}'] = np.cos(2 * np.pi * k * t / window)
    return df_feat

for col in FEATURE_COLS[:3]:  # İlk 3 sensör için
    df_features = add_fourier_features(df_features, col)

print(f'Fourier features added. Shape: {df_features.shape}')

## 10. Feature Importance — Mutual Information

Mutual information, her özelliğin hedef değişkenle (anomali) paylaştığı bilgi miktarını ölçer. Doğrusal olmayan ilişkileri de yakaladığı için korelasyondan daha güçlüdür.

In [ ]:
# Tüm sayısal featureları al
X = df_features.select_dtypes(include=[np.number]).drop(columns=[ANOMALY_COL], errors='ignore')
y = df_features[ANOMALY_COL].astype(int)

# NaN/inf temizle
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# Mutual information hesapla
print('Computing Mutual Information... (may take a few minutes)')
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

# Top 20 feature
fig, ax = plt.subplots(figsize=(12, 8))
top20 = mi_series.head(20)
colors = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(top20))]
top20.plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_title('Top 20 Features — Mutual Information with Anomaly Label', fontsize=14)
ax.set_xlabel('Mutual Information Score')
plt.tight_layout()
plt.savefig('results/feature_importance_mutual_info.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 most informative features:')
print(mi_series.head(10))

## 11. Class Imbalance Analizi

Anomali tespiti problemlerinde sınıf dengesizliği (class imbalance) kritik bir sorundur. Normal veri çok, anomali verisi az olduğunda modeller normal sınıfı overfit yapar.

In [ ]:
# Sınıf dağılımı
class_counts = df[ANOMALY_COL].value_counts()
class_pcts = df[ANOMALY_COL].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pasta grafiği
axes[0].pie(class_counts.values, labels=['Normal', 'Anomaly'],
            autopct='%1.2f%%', colors=['#2ecc71', '#e74c3c'],
            explode=(0, 0.05), startangle=90,
            textprops={'fontsize': 12})
axes[0].set_title('Class Distribution (Pie Chart)', fontsize=13)

# Bar grafiği
bars = axes[1].bar(['Normal (0)', 'Anomaly (1)'],
                   class_counts.values,
                   color=['#2ecc71', '#e74c3c'], alpha=0.85)
axes[1].set_title('Class Distribution (Count)', fontsize=13)
axes[1].set_ylabel('Count')
for bar, count, pct in zip(bars, class_counts.values, class_pcts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                f'{count:,}\n({pct:.2f}%)', ha='center', va='bottom', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results/class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

imbalance_ratio = class_counts.max() / class_counts.min()
print(f'Imbalance ratio (majority/minority): {imbalance_ratio:.1f}x')
print(f'\nRecommendation:')
if imbalance_ratio > 10:
    print('  → High imbalance! Use SMOTE, class_weight=balanced, or threshold tuning.')
elif imbalance_ratio > 3:
    print('  → Moderate imbalance. Consider class_weight=balanced.')
else:
    print('  → Mild imbalance. Standard models should work well.')

## 12. Feature Engineering Özeti ve Kaydetme

Mühendislik uygulanmış feature setini sonraki notebook'lar için kaydediyoruz.

In [ ]:
# Feature engineering özeti
original_features = len(FEATURE_COLS)
total_features = df_features.shape[1] - 1  # anomaly col hariç

print('=== FEATURE ENGINEERING SUMMARY ===')
print(f'Original features:      {original_features}')
print(f'Rolling features:       {original_features * len(WINDOWS) * 2}')
print(f'Lag features:           {original_features * len(LAGS)}')
print(f'Fourier features:       {min(3, len(FEATURE_COLS)) * 3 * 2}')
print(f'Total features:         {total_features}')
print(f'Dataset shape:          {df_features.shape}')
print(f'Anomaly ratio:          {df[ANOMALY_COL].mean()*100:.2f}%')

# Engineered features kaydet
os.makedirs('../data/processed', exist_ok=True)
df_features.to_csv('../data/processed/features_engineered.csv')
print('\nEngineered features saved to ../data/processed/features_engineered.csv')

print('\n✅ EDA & Feature Engineering Complete!')

## Özet

Bu notebook'ta gerçekleştirilen işlemler:

| Adım | İşlem | Çıktı |
|------|-------|-------|
| 1 | Dataset yükleme | Shape, dtypes, istatistikler |
| 2 | Zaman serisi görselleştirme | `sensor_time_series.png` |
| 3 | Korelasyon matrisi | `correlation_matrix.png` |
| 4 | Anomali timeline | `anomaly_timeline.png` |
| 5 | Rolling features (24, 48, 168h) | `rolling_features.png` |
| 6 | Lag features (1, 6, 12, 24h) | `lag_autocorrelation.png` |
| 7 | Fourier transform | `fft_analysis.png` |
| 8 | Mutual information | `feature_importance_mutual_info.png` |
| 9 | Class imbalance | `class_imbalance.png` |
| 10 | Feature kaydetme | `features_engineered.csv` |

**Sonraki Adım:** `02_Classical_ML_Baselines` notebook'una geçin.